# MetroRide Phase 1: a bronze/silver lakehouse with DuckLake

This tutorial builds the first two layers of the MetroRide lakehouse from the nightly booking-system exports in `bronze/`:

```
trips_2026-04.csv ┐                         ┌──────────────────┐   transform   ┌──────────────┐
trips_2026-05.csv ┴─ ingest (append) ──────▶│ bronze.trips_raw │ ────────────▶ │ silver.trips │
                                            └──────────────────┘               └──────────────┘
```

| Step | What you do |
|---|---|
| 1 | Create all tables |
| 2 | Ingest April data into bronze |
| 3 | Display bronze data |
| 4 | Transform bronze → silver |
| 5 | Display silver data |
| 6 | Show ingestion and snapshot information |
| 7 | Ingest May data into bronze (append mode, no duplicate rows) |
| 8 | Transform only the new bronze data → silver |
| 9 | Display data, catalog and snapshots of both layers |
| 10 | Time travel in silver |

**Tech stack**

* **DuckLake** is a lakehouse format: table data is stored as Parquet files, and all metadata (schemas, files, snapshots) lives in a SQL database called the *catalog*.
* **pyducklake** is a Python client for DuckLake with a pyiceberg-like API: `Catalog`, `create_table`, `append`, `scan`, `inspect`, time travel.
* **DuckDB** runs underneath pyducklake, and we use its SQL for the transformation logic.

**Install once** (in your terminal or a notebook cell):

```bash
pip install "pyducklake[pandas]" pytz
```

Run this notebook from the `phases/phase-1-batch/` folder.

## 0. Setup

We create (or recreate) the lakehouse folder `lakehouse/`:

* `lakehouse/metadata.ducklake` is the **catalog**, a DuckDB database file holding all DuckLake metadata.
* `lakehouse/data/` holds the **data files** (Parquet).

The folder is deleted at the start so that the tutorial can be re-run from a clean state.

In [1]:
import hashlib
import shutil
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
from pyducklake import (
    Catalog, Schema, required, optional,
    StringType, IntegerType, TimestampTZType, DecimalType, DateType,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

BKK = ZoneInfo("Asia/Bangkok")

PHASE_DIR = Path.cwd()                 # .../phases/phase-1-batch
SOURCE_DIR = PHASE_DIR / "bronze"      # nightly CSV exports from the booking system
LAKE_DIR = PHASE_DIR / "lakehouse"     # our DuckLake lakehouse
assert SOURCE_DIR.exists(), "Run this notebook from the phases/phase-1-batch/ folder"

# Start from an empty lakehouse every time the tutorial runs
if LAKE_DIR.exists():
    shutil.rmtree(LAKE_DIR)
LAKE_DIR.mkdir()

catalog = Catalog(
    "metroride",                                   # catalog name used in SQL: metroride.<schema>.<table>
    str(LAKE_DIR / "metadata.ducklake"),           # catalog database (DuckDB file)
    data_path=str(LAKE_DIR / "data"),              # where Parquet files are written
)

# Store table rows as Parquet files instead of inlining small inserts in the
# catalog database, so that we can see the files on disk.
catalog.set_option("data_inlining_row_limit", "0")

con = catalog.connection                           # the DuckDB connection behind pyducklake
con.execute("SET TimeZone = 'Asia/Bangkok'")       # show timestamps in Bangkok time


def sql(query: str) -> pd.DataFrame:
    # Run a SQL query on the lakehouse and return a pandas DataFrame.
    return con.sql(query).df()


def schema_df(table) -> pd.DataFrame:
    # Show a pyducklake table schema as a small DataFrame.
    return pd.DataFrame(
        [(f.name, str(f.field_type), "required" if f.required else "optional") for f in table.schema.fields],
        columns=["column", "type", "nullability"],
    )


print("Catalog :", (LAKE_DIR / "metadata.ducklake").relative_to(PHASE_DIR))
print("Data    :", (LAKE_DIR / "data").relative_to(PHASE_DIR))

Catalog : lakehouse/metadata.ducklake
Data    : lakehouse/data


## 1. Create all tables

DuckLake organises tables into **schemas** (pyducklake calls them *namespaces*). We create one namespace per layer.

**`bronze.trips_raw`** keeps the CSV values **exactly as received**: every source column is text. We add four ingest-metadata columns:

| Column | Purpose |
|---|---|
| `source_file` | Which CSV the row came from |
| `source_row_number` | Row position inside that CSV (1 = first data row) |
| `file_sha256` | Hash of the whole file, used to detect a file that is ingested twice |
| `ingested_at` | When the row was loaded |

**`silver.trips`** holds one **validated, typed** row per `trip_id`, following the phase-1 spec: timestamps with time zone, a `decimal(10,2)` fare, and two derived columns, `service_date` and `time_slot`.

In [2]:
catalog.create_namespace("bronze")
catalog.create_namespace("silver")

TRIP_COLUMNS = [
    "trip_id", "driver_id", "service_zone", "requested_at", "accepted_at",
    "picked_up_at", "completed_at", "cancelled_at", "status", "fare_thb",
]

bronze_schema = Schema.of(
    *[optional(col, StringType()) for col in TRIP_COLUMNS],   # raw source columns, all text
    required("source_file", StringType()),
    required("source_row_number", IntegerType()),
    required("file_sha256", StringType()),
    required("ingested_at", TimestampTZType()),
)
bronze = catalog.create_table(("bronze", "trips_raw"), bronze_schema)

silver_schema = Schema.of(
    required("trip_id", StringType()),
    required("driver_id", StringType()),
    required("service_zone", StringType()),
    required("status", StringType()),
    required("requested_at", TimestampTZType()),
    required("accepted_at", TimestampTZType()),
    optional("picked_up_at", TimestampTZType()),
    optional("completed_at", TimestampTZType()),
    optional("cancelled_at", TimestampTZType()),
    optional("fare_thb", DecimalType(10, 2)),
    required("service_date", DateType()),
    required("time_slot", StringType()),
)
silver = catalog.create_table(("silver", "trips"), silver_schema)

print("Namespaces:", catalog.list_namespaces())
for ns in ["bronze", "silver"]:
    print(f"Tables in {ns}:", catalog.list_tables(ns))

Namespaces: ['bronze', 'main', 'silver']
Tables in bronze: [('bronze', 'trips_raw')]
Tables in silver: [('silver', 'trips')]


In [3]:
schema_df(bronze)

,column,type,nullability
0,trip_id,StringType(),optional
1,driver_id,StringType(),optional
2,service_zone,StringType(),optional
3,requested_at,StringType(),optional
4,accepted_at,StringType(),optional
5,picked_up_at,StringType(),optional
6,completed_at,StringType(),optional
7,cancelled_at,StringType(),optional
8,status,StringType(),optional
9,fare_thb,StringType(),optional


In [4]:
schema_df(silver)

,column,type,nullability
0,trip_id,StringType(),required
1,driver_id,StringType(),required
2,service_zone,StringType(),required
3,status,StringType(),required
4,requested_at,TimestampTZType(),required
5,accepted_at,TimestampTZType(),required
6,picked_up_at,TimestampTZType(),optional
7,completed_at,TimestampTZType(),optional
8,cancelled_at,TimestampTZType(),optional
9,fare_thb,"DecimalType(precision=10, scale=2)",optional


## 2. Ingest April data into the bronze layer

`ingest_trips_csv()` loads one nightly CSV into `bronze.trips_raw` in **append mode**. Because a nightly job can be re-run by mistake, it has two guards against duplicate rows:

1. **File guard:** if a file with the same SHA-256 hash is already in bronze, skip the whole file.
2. **Row guard:** if the file is new but contains a `trip_id` already in bronze (for example a re-sent export), insert only the rows with new `trip_id`s. Repeated `trip_id`s inside the same file are also dropped.

Every column is read as **text** (`pa.string()`), so bronze never changes or rejects a source value. Blank cells become `NULL`.

The append runs inside a transaction with a **commit message**, which is stored on the DuckLake snapshot. We will see it again in Step 6.

In [5]:
def ingest_trips_csv(csv_path: Path) -> int:
    # Append one nightly trips CSV to bronze.trips_raw. Returns the number of rows inserted.
    file_bytes = csv_path.read_bytes()
    file_hash = hashlib.sha256(file_bytes).hexdigest()

    # Guard 1: the exact same file was ingested before
    if bronze.scan(f"file_sha256 = '{file_hash}'").count() > 0:
        print(f"SKIP {csv_path.name}: this file (sha256 {file_hash[:12]}...) is already in bronze")
        return 0

    # Read every column as text: bronze keeps source values as they arrived
    raw = pacsv.read_csv(
        csv_path,
        convert_options=pacsv.ConvertOptions(
            column_types={col: pa.string() for col in TRIP_COLUMNS},
            strings_can_be_null=True,          # blank cell -> NULL
        ),
    )
    if raw.column_names != TRIP_COLUMNS:
        raise ValueError(f"Unexpected header in {csv_path.name}: {raw.column_names}")

    # Add ingest metadata columns
    n = raw.num_rows
    batch = (
        raw.append_column("source_file", pa.array([csv_path.name] * n, pa.string()))
           .append_column("source_row_number", pa.array(range(1, n + 1), pa.int32()))
           .append_column("file_sha256", pa.array([file_hash] * n, pa.string()))
           .append_column("ingested_at", pa.array([datetime.now(BKK)] * n, pa.timestamp("us", tz="Asia/Bangkok")))
    )

    # Guard 2: keep only trip_ids that are not in bronze yet (and the first copy inside this file)
    con.register("incoming", batch)
    new_rows = con.sql('''
        SELECT *
        FROM incoming AS i
        WHERE NOT EXISTS (
            SELECT 1 FROM metroride.bronze.trips_raw AS b WHERE b.trip_id = i.trip_id
        )
        QUALIFY row_number() OVER (PARTITION BY trip_id ORDER BY source_row_number) = 1
        ORDER BY source_row_number
    ''').to_arrow_table()
    con.unregister("incoming")

    duplicates = n - new_rows.num_rows
    if new_rows.num_rows == 0:
        print(f"SKIP {csv_path.name}: all {n} rows are duplicates of rows already in bronze")
        return 0

    with catalog.begin_transaction() as txn:
        catalog.set_commit_message(f"bronze: ingest {csv_path.name}", author="phase-1 tutorial")
        txn.load_table(("bronze", "trips_raw")).append(new_rows)

    print(f"OK   {csv_path.name}: {n} rows read, {new_rows.num_rows} appended, {duplicates} duplicates skipped")
    return new_rows.num_rows

In [6]:
ingest_trips_csv(SOURCE_DIR / "trips_2026-04.csv")

OK   trips_2026-04.csv: 132 rows read, 132 appended, 0 duplicates skipped


132

## 3. Display data in the bronze layer

`table.scan()` builds a read query. You can add a filter, choose columns or set a limit, and then turn the result into pandas (`to_pandas()`), Arrow (`to_arrow()`), or just count the rows (`count()`).

In [7]:
print("Rows in bronze.trips_raw:", bronze.scan().count())
bronze.scan().with_limit(5).to_pandas()

Rows in bronze.trips_raw: 132


,trip_id,driver_id,service_zone,requested_at,accepted_at,picked_up_at,completed_at,cancelled_at,status,fare_thb,source_file,source_row_number,file_sha256,ingested_at
0,BKK-20260401-M1,D002,Asok,2026-04-01T08:03:00+07:00,2026-04-01T08:05:00+07:00,2026-04-01T08:11:00+07:00,2026-04-01T08:24:00+07:00,None,completed,148.00,trips_2026-04.csv,1,ee6286bcdafff7d1e5f0c771084ffc42bed1122db62155cad008a016...,2026-10-15 21:24:13.595302+07:00
1,BKK-20260401-M2,D003,Ari,2026-04-01T08:20:00+07:00,2026-04-01T08:23:00+07:00,2026-04-01T08:31:00+07:00,2026-04-01T08:45:00+07:00,None,completed,161.00,trips_2026-04.csv,2,ee6286bcdafff7d1e5f0c771084ffc42bed1122db62155cad008a016...,2026-10-15 21:24:13.595302+07:00
2,BKK-20260401-E1,D004,Ari,2026-04-01T18:03:00+07:00,2026-04-01T18:04:00+07:00,2026-04-01T18:15:00+07:00,2026-04-01T18:30:00+07:00,None,completed,166.00,trips_2026-04.csv,3,ee6286bcdafff7d1e5f0c771084ffc42bed1122db62155cad008a016...,2026-10-15 21:24:13.595302+07:00
3,BKK-20260401-E2,D005,Silom,2026-04-01T18:20:00+07:00,2026-04-01T18:22:00+07:00,2026-04-01T18:35:00+07:00,2026-04-01T18:51:00+07:00,None,completed,179.00,trips_2026-04.csv,4,ee6286bcdafff7d1e5f0c771084ffc42bed1122db62155cad008a016...,2026-10-15 21:24:13.595302+07:00
4,BKK-20260401-E3,D006,Siam,2026-04-01T18:37:00+07:00,2026-04-01T18:40:00+07:00,2026-04-01T18:50:00+07:00,2026-04-01T19:07:00+07:00,None,completed,160.00,trips_2026-04.csv,5,ee6286bcdafff7d1e5f0c771084ffc42bed1122db62155cad008a016...,2026-10-15 21:24:13.595302+07:00


Every source column is still text, and the blank cells of cancelled trips are `None` (NULL):

In [8]:
bronze.scan("status = 'cancelled'").select(
    "trip_id", "status", "picked_up_at", "completed_at", "cancelled_at", "fare_thb", "source_row_number"
).to_pandas()

,trip_id,status,picked_up_at,completed_at,cancelled_at,fare_thb,source_row_number
0,BKK-20260405-M1,cancelled,None,None,2026-04-05T08:06:00+07:00,None,18
1,BKK-20260406-E3,cancelled,None,None,2026-04-06T18:40:00+07:00,None,27
2,BKK-20260409-E1,cancelled,None,None,2026-04-09T18:08:00+07:00,None,38
3,BKK-20260413-E2,cancelled,None,None,2026-04-13T18:23:00+07:00,None,57
4,BKK-20260415-M2,cancelled,None,None,2026-04-15T08:23:00+07:00,None,63
5,BKK-20260418-M1,cancelled,None,None,2026-04-18T08:07:00+07:00,None,76
6,BKK-20260422-M2,cancelled,None,None,2026-04-22T08:25:00+07:00,None,95
7,BKK-20260425-M3,cancelled,None,None,2026-04-25T08:40:00+07:00,None,108


## 4. Transform data from bronze to silver

The transformation is written in SQL and runs in three stages:

1. **Pick new rows:** take the bronze rows whose `trip_id` is not in silver yet. This makes the transformation **incremental** and **idempotent**: running it twice inserts nothing the second time.
2. **Parse types:** turn text into `TIMESTAMPTZ` and `DECIMAL(10,2)`. `TRY_CAST` returns `NULL` instead of failing on a bad value. Then derive `service_date` and `time_slot` from `requested_at` in Bangkok time.
3. **Validate:** keep only rows that satisfy the phase-1 rules. Rows that fail stay in bronze but do not reach silver, so they are counted as "not yet in silver" on every run until the source is fixed.

| Status | Rule |
|---|---|
| all | `trip_id`, `driver_id`, `requested_at`, `accepted_at` present; zone in Siam/Asok/Ari/Silom; `requested_at <= accepted_at`; request at 08:xx or 18:xx |
| `completed` | `accepted_at <= picked_up_at <= completed_at`; fare present and `>= 0`; no `cancelled_at` |
| `cancelled` | `accepted_at <= cancelled_at`; no pickup, completion or fare |

In [9]:
TRANSFORM_SQL = '''
WITH new_rows AS (               -- 1. bronze rows that are not in silver yet
    SELECT b.*
    FROM metroride.bronze.trips_raw AS b
    WHERE NOT EXISTS (
        SELECT 1 FROM metroride.silver.trips AS s WHERE s.trip_id = b.trip_id
    )
),
typed AS (                       -- 2. parse text into proper types
    SELECT
        trim(trip_id)                         AS trip_id,
        trim(driver_id)                       AS driver_id,
        trim(service_zone)                    AS service_zone,
        lower(trim(status))                   AS status,
        TRY_CAST(requested_at AS TIMESTAMPTZ) AS requested_at,
        TRY_CAST(accepted_at  AS TIMESTAMPTZ) AS accepted_at,
        TRY_CAST(picked_up_at AS TIMESTAMPTZ) AS picked_up_at,
        TRY_CAST(completed_at AS TIMESTAMPTZ) AS completed_at,
        TRY_CAST(cancelled_at AS TIMESTAMPTZ) AS cancelled_at,
        TRY_CAST(fare_thb AS DECIMAL(10, 2))  AS fare_thb,
        ingested_at
    FROM new_rows
),
derived AS (                     --    derive date and slot in Bangkok local time
    SELECT
        *,
        CAST(timezone('Asia/Bangkok', requested_at) AS DATE) AS service_date,
        CASE hour(timezone('Asia/Bangkok', requested_at))
            WHEN 8  THEN 'morning'
            WHEN 18 THEN 'evening'
        END AS time_slot
    FROM typed
)
SELECT                           -- 3. validate; column order matches silver.trips
    trip_id, driver_id, service_zone, status,
    requested_at, accepted_at, picked_up_at, completed_at, cancelled_at,
    fare_thb, service_date, time_slot
FROM derived
WHERE trip_id IS NOT NULL
  AND driver_id IS NOT NULL
  AND service_zone IN ('Siam', 'Asok', 'Ari', 'Silom')
  AND requested_at IS NOT NULL
  AND accepted_at IS NOT NULL
  AND requested_at <= accepted_at
  AND time_slot IS NOT NULL
  AND (
        (status = 'completed'
         AND picked_up_at IS NOT NULL AND completed_at IS NOT NULL
         AND accepted_at <= picked_up_at AND picked_up_at <= completed_at
         AND fare_thb IS NOT NULL AND fare_thb >= 0
         AND cancelled_at IS NULL)
     OR (status = 'cancelled'
         AND cancelled_at IS NOT NULL AND accepted_at <= cancelled_at
         AND picked_up_at IS NULL AND completed_at IS NULL AND fare_thb IS NULL)
  )
QUALIFY row_number() OVER (PARTITION BY trip_id ORDER BY ingested_at) = 1   -- one row per trip_id
ORDER BY requested_at
'''


def transform_bronze_to_silver() -> int:
    # Move new, valid bronze rows into silver.trips. Returns the number of rows appended.
    pending = sql('''
        SELECT count(*) AS n FROM metroride.bronze.trips_raw AS b
        WHERE NOT EXISTS (SELECT 1 FROM metroride.silver.trips AS s WHERE s.trip_id = b.trip_id)
    ''').n[0]
    clean = con.sql(TRANSFORM_SQL).to_arrow_table()

    if clean.num_rows == 0:
        print(f"Nothing to do: {pending} bronze rows not yet in silver, 0 valid rows to append")
        return 0

    with catalog.begin_transaction() as txn:
        catalog.set_commit_message(f"silver: transform {clean.num_rows} new trips", author="phase-1 tutorial")
        txn.load_table(("silver", "trips")).append(clean)

    print(f"{pending} bronze rows not yet in silver -> {clean.num_rows} appended, {pending - clean.num_rows} rejected by validation")
    return clean.num_rows

In [10]:
transform_bronze_to_silver()

# Remember this moment so that we can time travel back to it in Step 10
silver_snapshot_april = silver.current_snapshot().snapshot_id
time_after_april = datetime.now(BKK)
print("Silver snapshot after April:", silver_snapshot_april)

132 bronze rows not yet in silver -> 132 appended, 0 rejected by validation
Silver snapshot after April: 6


## 5. Display data in the silver layer

The values now have real types: timestamps carry the `+07:00` offset, fares are decimals, and `service_date` and `time_slot` have been derived.

In [11]:
print("Rows in silver.trips:", silver.scan().count())
silver.scan().with_limit(5).to_pandas()

Rows in silver.trips: 132


,trip_id,driver_id,service_zone,status,requested_at,accepted_at,picked_up_at,completed_at,cancelled_at,fare_thb,service_date,time_slot
0,BKK-20260401-M1,D002,Asok,completed,2026-04-01 08:03:00+07:00,2026-04-01 08:05:00+07:00,2026-04-01 08:11:00+07:00,2026-04-01 08:24:00+07:00,NaT,148.0,2026-04-01,morning
1,BKK-20260401-M2,D003,Ari,completed,2026-04-01 08:20:00+07:00,2026-04-01 08:23:00+07:00,2026-04-01 08:31:00+07:00,2026-04-01 08:45:00+07:00,NaT,161.0,2026-04-01,morning
2,BKK-20260401-E1,D004,Ari,completed,2026-04-01 18:03:00+07:00,2026-04-01 18:04:00+07:00,2026-04-01 18:15:00+07:00,2026-04-01 18:30:00+07:00,NaT,166.0,2026-04-01,evening
3,BKK-20260401-E2,D005,Silom,completed,2026-04-01 18:20:00+07:00,2026-04-01 18:22:00+07:00,2026-04-01 18:35:00+07:00,2026-04-01 18:51:00+07:00,NaT,179.0,2026-04-01,evening
4,BKK-20260401-E3,D006,Siam,completed,2026-04-01 18:37:00+07:00,2026-04-01 18:40:00+07:00,2026-04-01 18:50:00+07:00,2026-04-01 19:07:00+07:00,NaT,160.0,2026-04-01,evening


The Arrow schema shows the exact stored types. (pandas would show the decimal fare as `float64` and the date as `datetime64`.)

In [12]:
silver.scan().to_arrow().schema

trip_id: string
driver_id: string
service_zone: string
status: string
requested_at: timestamp[us, tz=Asia/Bangkok]
accepted_at: timestamp[us, tz=Asia/Bangkok]
picked_up_at: timestamp[us, tz=Asia/Bangkok]
completed_at: timestamp[us, tz=Asia/Bangkok]
cancelled_at: timestamp[us, tz=Asia/Bangkok]
fare_thb: decimal128(10, 2)
service_date: date32[day]
time_slot: string

A quick sanity check of the content. `scan().to_duckdb()` returns a DuckDB relation, so you can keep querying it with SQL-style methods:

In [13]:
silver.scan().to_duckdb().aggregate(
    "time_slot, status, count(*) AS trips, sum(fare_thb) AS fare_thb", "time_slot, status"
).order("time_slot, status").df()

,time_slot,status,trips,fare_thb
0,evening,cancelled,3,NaN
1,evening,completed,63,11414.0
2,morning,cancelled,5,NaN
3,morning,completed,61,11161.0


## 6. Show ingestion and snapshot information

**Ingestion information** comes from the metadata columns we added in bronze: one row per ingested file.

In [14]:
sql('''
    SELECT source_file,
           left(file_sha256, 16) || '...' AS file_sha256,
           min(ingested_at)              AS ingested_at,
           count(*)                      AS rows,
           min(source_row_number)        AS first_row,
           max(source_row_number)        AS last_row
    FROM metroride.bronze.trips_raw
    GROUP BY ALL
    ORDER BY ingested_at
''')

,source_file,file_sha256,ingested_at,rows,first_row,last_row
0,trips_2026-04.csv,ee6286bcdafff7d1...,2026-10-15 21:24:13.595302+07:00,132,1,132


**Snapshots.** Every committed change to a DuckLake catalog (create schema, create table, insert, ...) creates a new **snapshot** with an increasing `snapshot_id`. Snapshots are **catalog-wide**: one snapshot can touch several tables. The `changes` column says what happened in each one, and `commit_message` shows the messages we set.

`tables_inserted_into` lists **table IDs**, not names. Here ID 3 is `bronze.trips_raw` and ID 4 is `silver.trips`; Step 9 shows how to look this up.

In [15]:
def snapshots_df() -> pd.DataFrame:
    df = bronze.inspect().snapshots().to_pandas()
    return df[["snapshot_id", "snapshot_time", "schema_version", "changes", "author", "commit_message"]]


snapshots_df()

,snapshot_id,snapshot_time,schema_version,changes,author,commit_message
0,0,2026-10-15 21:24:13.464697+07:00,0,"[(schemas_created, [main])]",NaN,NaN
1,1,2026-10-15 21:24:13.502115+07:00,1,"[(schemas_created, [bronze])]",NaN,NaN
2,2,2026-10-15 21:24:13.511201+07:00,2,"[(schemas_created, [silver])]",NaN,NaN
3,3,2026-10-15 21:24:13.520691+07:00,3,"[(tables_created, [bronze.trips_raw])]",NaN,NaN
4,4,2026-10-15 21:24:13.541827+07:00,4,"[(tables_created, [silver.trips])]",NaN,NaN
5,5,2026-10-15 21:24:15.916152+07:00,4,"[(tables_inserted_into, [3])]",phase-1 tutorial,bronze: ingest trips_2026-04.csv
6,6,2026-10-15 21:24:15.977966+07:00,4,"[(tables_inserted_into, [4])]",phase-1 tutorial,silver: transform 132 new trips


Each insert wrote a **Parquet file**. `inspect().files()` lists the data files of a table:

In [16]:
for t in [bronze, silver]:
    files = t.inspect().files().to_pandas()
    print(f"{t.namespace}.{t.name}: {len(files)} data file(s)")
    for f in files.itertuples():
        print("   ", Path(f.data_file).relative_to(LAKE_DIR), f"({f.data_file_size_bytes:,} bytes)")

bronze.trips_raw: 1 data file(s)
    data/bronze/trips_raw/ducklake-01a13ff3-0e33-7d7f-93e9-83e79409aa58.parquet (8,250 bytes)
silver.trips: 1 data file(s)
    data/silver/trips/ducklake-01a13ff3-0e6d-7ff9-b683-8cc6e469ffd9.parquet (7,179 bytes)


## 7. Ingest May data into bronze (append mode, no duplicate rows)

First, the normal nightly load of the May file:

In [17]:
ingest_trips_csv(SOURCE_DIR / "trips_2026-05.csv")

OK   trips_2026-05.csv: 137 rows read, 137 appended, 0 duplicates skipped


137

Now we test the duplicate guards.

**Replay:** the scheduler re-runs the job over every file in the folder. Both files are already in bronze, so the **file guard** skips them.

In [18]:
for csv_file in sorted(SOURCE_DIR.glob("trips_*.csv")):
    ingest_trips_csv(csv_file)

SKIP trips_2026-04.csv: this file (sha256 ee6286bcdaff...) is already in bronze
SKIP trips_2026-05.csv: this file (sha256 d9b4c1ca04b9...) is already in bronze


**Re-sent export:** the booking system sends a *different* file that repeats three May trips. Its hash is new, so the file guard lets it through, but the **row guard** finds that every `trip_id` is already in bronze.

In [19]:
resend_dir = LAKE_DIR / "landing"
resend_dir.mkdir(exist_ok=True)
may_lines = (SOURCE_DIR / "trips_2026-05.csv").read_text().splitlines()
resend_file = resend_dir / "trips_2026-05_resend.csv"
resend_file.write_text("\n".join(may_lines[:4]) + "\n")   # header + first 3 May trips

ingest_trips_csv(resend_file)

SKIP trips_2026-05_resend.csv: all 3 rows are duplicates of rows already in bronze


0

In [20]:
sql('''
    SELECT count(*)                           AS bronze_rows,
           count(DISTINCT trip_id)            AS distinct_trip_ids,
           count(*) - count(DISTINCT trip_id) AS duplicate_rows
    FROM metroride.bronze.trips_raw
''')

,bronze_rows,distinct_trip_ids,duplicate_rows
0,269,269,0


## 8. Transform the new bronze data into silver

The same function as in Step 4. Because it only picks bronze rows whose `trip_id` is not in silver, it processes just the 137 May trips. Running it a second time does nothing.

In [21]:
transform_bronze_to_silver()
silver_snapshot_may = silver.current_snapshot().snapshot_id
print("Silver snapshot after May:", silver_snapshot_may)

137 bronze rows not yet in silver -> 137 appended, 0 rejected by validation
Silver snapshot after May: 8


In [22]:
transform_bronze_to_silver()   # run again: idempotent, nothing new to insert

Nothing to do: 0 bronze rows not yet in silver, 0 valid rows to append


0

## 9. Display data, catalogs and snapshots of both layers

### 9.1 Data

Row counts per month in each layer:

In [23]:
sql('''
    SELECT 'bronze.trips_raw' AS table_name, substr(requested_at, 1, 7) AS month, count(*) AS rows
    FROM metroride.bronze.trips_raw GROUP BY ALL
    UNION ALL
    SELECT 'silver.trips', strftime(service_date, '%Y-%m'), count(*)
    FROM metroride.silver.trips GROUP BY ALL
    ORDER BY table_name, month
''')

,table_name,month,rows
0,bronze.trips_raw,2026-04,132
1,bronze.trips_raw,2026-05,137
2,silver.trips,2026-04,132
3,silver.trips,2026-05,137


The same May trip in both layers: text in bronze, typed and enriched in silver.

In [24]:
bronze.scan("trip_id = 'BKK-20260501-M1'").to_pandas().T

,0
trip_id,BKK-20260501-M1
driver_id,D002
service_zone,Asok
requested_at,2026-05-01T08:03:00+07:00
accepted_at,2026-05-01T08:05:00+07:00
picked_up_at,2026-05-01T08:11:00+07:00
completed_at,2026-05-01T08:24:00+07:00
cancelled_at,None
status,completed
fare_thb,148.00


In [25]:
silver.scan("trip_id = 'BKK-20260501-M1'").to_pandas().T

,0
trip_id,BKK-20260501-M1
driver_id,D002
service_zone,Asok
status,completed
requested_at,2026-05-01 08:03:00+07:00
accepted_at,2026-05-01 08:05:00+07:00
picked_up_at,2026-05-01 08:11:00+07:00
completed_at,2026-05-01 08:24:00+07:00
cancelled_at,NaT
fare_thb,148.0


### 9.2 Catalog

The catalog knows which namespaces and tables exist, their schemas, and their files. `ducklake_table_info()` summarises each table:

In [26]:
for ns in catalog.list_namespaces():
    print(f"{ns:8s} -> {catalog.list_tables(ns)}")

sql('''
    SELECT s.schema_name, i.table_name, i.table_id, i.file_count, i.file_size_bytes
    FROM ducklake_table_info('metroride') AS i
    JOIN __ducklake_metadata_metroride.ducklake_schema AS s USING (schema_id)
    ORDER BY ALL
''')

bronze   -> [('bronze', 'trips_raw')]
main     -> []
silver   -> [('silver', 'trips')]


,schema_name,table_name,table_id,file_count,file_size_bytes
0,bronze,trips_raw,3,2,16674
1,silver,trips,4,2,14529


In [27]:
for t in [bronze, silver]:
    files = t.inspect().files().to_pandas()
    print(f"{t.namespace}.{t.name}: {len(files)} data file(s)")
    for f in files.itertuples():
        print("   ", Path(f.data_file).relative_to(LAKE_DIR), f"({f.data_file_size_bytes:,} bytes)")

bronze.trips_raw: 2 data file(s)
    data/bronze/trips_raw/ducklake-01a13ff3-0e33-7d7f-93e9-83e79409aa58.parquet (8,250 bytes)
    data/bronze/trips_raw/ducklake-01a13ff3-0ec7-7fae-901d-213fff5f478e.parquet (8,424 bytes)
silver.trips: 2 data file(s)
    data/silver/trips/ducklake-01a13ff3-0e6d-7ff9-b683-8cc6e469ffd9.parquet (7,179 bytes)
    data/silver/trips/ducklake-01a13ff3-0ef3-7d0c-b16a-83164ce7a509.parquet (7,350 bytes)


The DuckLake catalog is **just a set of ordinary SQL tables** inside `metadata.ducklake`. pyducklake attaches them under the name `__ducklake_metadata_metroride`, so you can query them directly. The most important ones are:

| Metadata table | Contents |
|---|---|
| `ducklake_snapshot` | One row per snapshot: id, time, schema version |
| `ducklake_snapshot_changes` | What each snapshot changed, plus author and commit message |
| `ducklake_schema`, `ducklake_table`, `ducklake_column` | Namespaces, tables and columns, each valid from a `begin_snapshot` to an `end_snapshot` |
| `ducklake_data_file` | Every Parquet file: table, path, row count, and the snapshot that added it |

Here are the data files and the snapshot that added each one:

In [28]:
sql('''
    SELECT s.schema_name || '.' || t.table_name AS table_name,
           f.data_file_id,
           f.path,
           f.record_count,
           f.begin_snapshot AS added_in_snapshot
    FROM __ducklake_metadata_metroride.ducklake_data_file AS f
    JOIN __ducklake_metadata_metroride.ducklake_table  AS t USING (table_id)
    JOIN __ducklake_metadata_metroride.ducklake_schema AS s USING (schema_id)
    ORDER BY f.begin_snapshot
''')

,table_name,data_file_id,path,record_count,added_in_snapshot
0,bronze.trips_raw,0,ducklake-01a13ff3-0e33-7d7f-93e9-83e79409aa58.parquet,132,5
1,silver.trips,1,ducklake-01a13ff3-0e6d-7ff9-b683-8cc6e469ffd9.parquet,132,6
2,bronze.trips_raw,2,ducklake-01a13ff3-0ec7-7fae-901d-213fff5f478e.parquet,137,7
3,silver.trips,3,ducklake-01a13ff3-0ef3-7d0c-b16a-83164ce7a509.parquet,137,8


### 9.3 Snapshots

All snapshots of the catalog. Notice that the re-runs in Steps 7 and 8 created **no snapshots**, because nothing was written.

In [29]:
snapshots_df()

,snapshot_id,snapshot_time,schema_version,changes,author,commit_message
0,0,2026-10-15 21:24:13.464697+07:00,0,"[(schemas_created, [main])]",NaN,NaN
1,1,2026-10-15 21:24:13.502115+07:00,1,"[(schemas_created, [bronze])]",NaN,NaN
2,2,2026-10-15 21:24:13.511201+07:00,2,"[(schemas_created, [silver])]",NaN,NaN
3,3,2026-10-15 21:24:13.520691+07:00,3,"[(tables_created, [bronze.trips_raw])]",NaN,NaN
4,4,2026-10-15 21:24:13.541827+07:00,4,"[(tables_created, [silver.trips])]",NaN,NaN
5,5,2026-10-15 21:24:15.916152+07:00,4,"[(tables_inserted_into, [3])]",phase-1 tutorial,bronze: ingest trips_2026-04.csv
6,6,2026-10-15 21:24:15.977966+07:00,4,"[(tables_inserted_into, [4])]",phase-1 tutorial,silver: transform 132 new trips
7,7,2026-10-15 21:24:16.068918+07:00,4,"[(tables_inserted_into, [3])]",phase-1 tutorial,bronze: ingest trips_2026-05.csv
8,8,2026-10-15 21:24:16.112972+07:00,4,"[(tables_inserted_into, [4])]",phase-1 tutorial,silver: transform 137 new trips


Snapshots are catalog-wide, so to see the history of **one layer** we keep only the snapshots that added data to that table:

In [30]:
sql('''
    SELECT s.schema_name || '.' || t.table_name AS table_name,
           sn.snapshot_id,
           sn.snapshot_time,
           f.record_count AS rows_added,
           c.commit_message
    FROM __ducklake_metadata_metroride.ducklake_data_file AS f
    JOIN __ducklake_metadata_metroride.ducklake_table  AS t USING (table_id)
    JOIN __ducklake_metadata_metroride.ducklake_schema AS s USING (schema_id)
    JOIN __ducklake_metadata_metroride.ducklake_snapshot AS sn ON sn.snapshot_id = f.begin_snapshot
    LEFT JOIN __ducklake_metadata_metroride.ducklake_snapshot_changes AS c ON c.snapshot_id = sn.snapshot_id
    ORDER BY table_name, sn.snapshot_id
''')

,table_name,snapshot_id,snapshot_time,rows_added,commit_message
0,bronze.trips_raw,5,2026-10-15 21:24:15.916152+07:00,132,bronze: ingest trips_2026-04.csv
1,bronze.trips_raw,7,2026-10-15 21:24:16.068918+07:00,137,bronze: ingest trips_2026-05.csv
2,silver.trips,6,2026-10-15 21:24:15.977966+07:00,132,silver: transform 132 new trips
3,silver.trips,8,2026-10-15 21:24:16.112972+07:00,137,silver: transform 137 new trips


## 10. Time travel in the silver layer

Because DuckLake never overwrites old files and remembers every snapshot, we can read **any table as it was at any earlier snapshot**.

### 10.1 By snapshot ID

`scan().with_snapshot(id)` reads the table as of that snapshot.

In [31]:
print(f"silver.trips at snapshot {silver_snapshot_april} (after April):", silver.scan().with_snapshot(silver_snapshot_april).count(), "rows")
print(f"silver.trips at snapshot {silver_snapshot_may} (after May):  ", silver.scan().with_snapshot(silver_snapshot_may).count(), "rows")
print("silver.trips now:", silver.scan().count(), "rows")

silver.trips at snapshot 6 (after April): 132 rows
silver.trips at snapshot 8 (after May):   269 rows
silver.trips now: 269 rows


### 10.2 By timestamp

`scan().with_timestamp(t)` reads the table as it was at a point in time. We saved `time_after_april` right after the April transform:

In [32]:
print("time_after_april =", time_after_april)
silver_then = silver.scan().with_timestamp(time_after_april).to_pandas()
print("Rows:", len(silver_then), "| service_date range:", silver_then.service_date.min(), "to", silver_then.service_date.max())

time_after_april = 2026-10-15 21:24:16.004286+07:00
Rows: 132 | service_date range: 2026-04-01 00:00:00 to 2026-04-30 00:00:00


### 10.3 In SQL

The same feature in SQL is `AT (VERSION => ...)` or `AT (TIMESTAMP => ...)`. Here we compare monthly figures in the old and current versions side by side:

In [33]:
sql(f'''
    SELECT 'snapshot {silver_snapshot_april} (after April)' AS version,
           strftime(service_date, '%Y-%m') AS month,
           count(*) AS requests,
           count(*) FILTER (status = 'completed') AS completed,
           sum(fare_thb) AS gross_fare_thb
    FROM metroride.silver.trips AT (VERSION => {silver_snapshot_april})
    GROUP BY ALL
    UNION ALL
    SELECT 'current',
           strftime(service_date, '%Y-%m'),
           count(*),
           count(*) FILTER (status = 'completed'),
           sum(fare_thb)
    FROM metroride.silver.trips
    GROUP BY ALL
    ORDER BY version DESC, month
''')

,version,month,requests,completed,gross_fare_thb
0,snapshot 6 (after April),2026-04,132,124,22575.0
1,current,2026-04,132,124,22575.0
2,current,2026-05,137,127,23089.0


### 10.4 What changed between two snapshots?

`table_insertions()` returns the rows inserted between two snapshots (change data capture). Starting just after the April snapshot, we get exactly the May trips:

In [34]:
changes = silver.table_insertions(start_snapshot=silver_snapshot_april + 1, end_snapshot=silver_snapshot_may)
inserted = changes.to_pandas()
print("Rows inserted after snapshot", silver_snapshot_april, ":", len(inserted))
inserted[["trip_id", "service_date", "time_slot", "status", "fare_thb"]].head()

Rows inserted after snapshot 6 : 137


,trip_id,service_date,time_slot,status,fare_thb
0,BKK-20260501-M1,2026-05-01,morning,completed,148.00
1,BKK-20260501-M2,2026-05-01,morning,completed,161.00
2,BKK-20260501-E1,2026-05-01,evening,completed,166.00
3,BKK-20260501-E2,2026-05-01,evening,completed,179.00
4,BKK-20260501-E3,2026-05-01,evening,completed,160.00


### 10.5 Before the data existed

Going back to the snapshot in which `silver.trips` was created shows an empty table:

In [35]:
silver_created = sql('''
    SELECT min(snapshot_id) AS id FROM metroride.snapshots()
    WHERE list_contains(map_extract(changes, 'tables_created')[1], 'silver.trips')
''').id[0]
print(f"silver.trips at snapshot {silver_created} (table just created):", silver.scan().with_snapshot(silver_created).count(), "rows")

silver.trips at snapshot 4 (table just created): 0 rows


## Summary

* **Bronze** stores every source value as text, plus ingest metadata (`source_file`, `source_row_number`, `file_sha256`, `ingested_at`). A file guard and a row guard keep re-runs and re-sent files from creating duplicate rows.
* **Silver** stores one typed, validated row per `trip_id`. The transformation is incremental (only bronze rows not yet in silver) and idempotent (running it again inserts nothing).
* **DuckLake** keeps metadata in a SQL catalog and data in Parquet files. Every committed write creates a snapshot, which enables time travel by snapshot ID or timestamp, and change queries between snapshots.

**Try it yourself**

1. Make a copy of `trips_2026-04.csv` with one bad row (for example a completed trip with no fare), ingest it, and check that the row reaches bronze but not silver.
2. Use time travel on `bronze.trips_raw` to show the table before the May file was ingested.
3. Build `gold.daily_service_performance` from `silver.trips` as described in the phase-1 README.